In [ ]:
import os
import json
import boto3
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import mlflow
import mlflow.pytorch
from botocore.exceptions import ClientError
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from pydantic import BaseModel, Field, field_validator
from typing import List, Dict, Optional, Tuple, Any

# --- 1. Configuration de l'Environnement ---
class Config:
    # Chemins
    CSV_PATH = "../annotations.csv"
    IMG_DIR = "../dataset_prepro/"
    
    # MLflow & MinIO
    MLFLOW_URI = 'http://localhost:5000'
    S3_ENDPOINT = 'http://localhost:9000'
    AWS_ID = 'minioadmin'
    AWS_KEY = 'minioadmin'
    EXPERIMENT_NAME = "Classification"
    
    # Hardware
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Setup Environment Variables
    @classmethod
    def setup_env(cls):
        os.environ['MLFLOW_TRACKING_URI'] = cls.MLFLOW_URI
        os.environ['MLFLOW_S3_ENDPOINT_URL'] = cls.S3_ENDPOINT
        os.environ['AWS_ACCESS_KEY_ID'] = cls.AWS_ID
        os.environ['AWS_SECRET_ACCESS_KEY'] = cls.AWS_KEY
        os.environ['MLFLOW_S3_IGNORE_TLS'] = 'true'
        os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
        print(f"🔥 Device: {cls.DEVICE}")

Config.setup_env()

c:\Users\Camille\Projects\projet-mlops\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


🔥 Device: cpu


In [3]:
# --- 2. Vérification Infrastructure ---
def ensure_bucket_exists(bucket_name="mlflow"):
    """Vérifie et crée le bucket S3 si nécessaire."""
    s3 = boto3.client('s3', 
                      endpoint_url=Config.S3_ENDPOINT,
                      aws_access_key_id=Config.AWS_ID,
                      aws_secret_access_key=Config.AWS_KEY)
    try:
        s3.head_bucket(Bucket=bucket_name)
    except ClientError as e:
        if int(e.response['Error']['Code']) == 404:
            print(f"⚠️ Bucket '{bucket_name}' introuvable. Création...")
            s3.create_bucket(Bucket=bucket_name)
            print("✅ Bucket créé.")
        else:
            raise

try:
    ensure_bucket_exists()
except Exception as e:
    print(f"❌ Erreur MinIO: {e}")

In [4]:
# --- 3. Gestion des Données ---
class DatasetFaces(Dataset):
    def __init__(self, csv_file: str, img_dir: str, transform=None):
        self.annotations = pd.read_csv(csv_file, index_col='filename')
        self.img_names = self.annotations.index.tolist()
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self): 
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, f"{img_name}.png") # Ou .jpg selon dataset
        
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (64, 64)) # Fallback image noire
            
        # Conversion des labels
        labels = torch.tensor(
            self.annotations.loc[img_name].values.astype(float), 
            dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, labels

def get_data_loaders(batch_size: int, split_ratio=0.8):
    """Génère les dataloaders train/test."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    dataset = DatasetFaces(Config.CSV_PATH, Config.IMG_DIR, transform=transform)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size
    
    train_ds, test_ds = random_split(
        dataset, [train_size, test_size], 
        generator=torch.Generator().manual_seed(42)
    )
    
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    )

In [5]:
# --- 4. Définition de l'Architecture ---

# A. Configuration (Validation Pydantic)
class TreeNodeConfig(BaseModel):
    layers: List[int] = Field(description="Liste des tailles de couches [512, 256]")
    dropout: float = Field(default=0.0, ge=0.0, le=1.0)
    branches: Dict[str, 'TreeNodeConfig'] = Field(default_factory=dict)
    tasks: Dict[str, int] = Field(default_factory=dict)

TreeNodeConfig.model_rebuild()

# B. Module Récursif (L'arbre)
class DynamicTreeBranch(nn.Module):
    def __init__(self, in_features: int, config: TreeNodeConfig):
        super().__init__()
        
        # 1. Tronc local
        layers = []
        current_size = in_features
        for i, hidden_size in enumerate(config.layers):
            layers.extend([
                nn.Linear(current_size, hidden_size),
                nn.ReLU(),
            ])
            if config.dropout > 0:
                layers.append(nn.Dropout(config.dropout))
            current_size = hidden_size
        self.local_layers = nn.Sequential(*layers)
        
        # 2. Sous-branches
        self.branches = nn.ModuleDict({
            name: DynamicTreeBranch(current_size, cfg) 
            for name, cfg in config.branches.items()
        })
        
        # 3. Têtes de sortie (Feuilles)
        self.tasks = nn.ModuleDict({
            name: nn.Linear(current_size, num_classes) 
            for name, num_classes in config.tasks.items()
        })

    def forward(self, x):
        x = self.local_layers(x)
        results = {}
        # Récupération récursive
        for branch in self.branches.values():
            results.update(branch(x))
        # Calcul local
        for name, head in self.tasks.items():
            results[name] = head(x)
        return results

# C. Modèle Global (CNN + Arbre)
class CNN(nn.Module):
    def __init__(self, filters_list: List[int], tree_config: TreeNodeConfig):
        super().__init__()
        self.filters_list = filters_list
        self.tree_structure_dict = tree_config.model_dump()
        
        # Feature Extractor
        layers = []
        in_c = 3
        for i, out_c in enumerate(filters_list):
            layers.extend([
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU()
            ])
            if i > 0: layers.append(nn.MaxPool2d(2))
            in_c = out_c
            
        self.conv = nn.Sequential(*layers)
        
        # Calcul dynamique de la taille
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 64, 64)
            self.flat_size = self.conv(dummy).view(1, -1).shape[1]
            
        # Arbre Fully Connected
        self.tree = DynamicTreeBranch(self.flat_size, tree_config)

    def forward(self, x):
        features = self.conv(x)
        flattened = features.view(features.size(0), -1)
        return self.tree(flattened)

In [6]:
# --- 5. Factory d'Architecture ---
def build_tree_config(params: Dict[str, Any]) -> TreeNodeConfig:
    """Construit la config de l'arbre selon le blueprint choisi."""
    blueprint = params['blueprint']
    dropout = params['dropout']
    
    # Définition des sorties (Immuable)
    TASKS_FACE = {"barbe": 1, "moustache": 1, "lunettes": 1}
    TASKS_HAIR = {"taille_cheveux": 3, "couleur_cheveux": 5}
    
    if blueprint == 'hierarchical':
        # Structure: Tronc -> [Visage] + [Cheveux -> (Optionnel: Split)]
        
        # Gestion de la sous-branche optionnelle cheveux
        hair_sub_branches = {}
        hair_tasks = TASKS_HAIR
        
        if params.get('use_split_hair', False):
            hair_tasks = {} # Les tâches sont déléguées plus bas
            hair_sub_branches["split_hair"] = TreeNodeConfig(
                layers=params['hair_split_layers'],
                tasks=TASKS_HAIR
            )

        return TreeNodeConfig(
            layers=params['trunk_layers'],
            dropout=dropout,
            branches={
                "visage": TreeNodeConfig(layers=params['visage_layers'], tasks=TASKS_FACE),
                "cheveux": TreeNodeConfig(
                    layers=params['hair_layers'],
                    branches=hair_sub_branches,
                    tasks=hair_tasks
                )
            }
        )

    elif blueprint == 'shared_trunk':
        # Structure: Gros Tronc -> Tout le monde
        return TreeNodeConfig(
            layers=params['shared_layers'],
            dropout=dropout,
            branches={
                "heads": TreeNodeConfig(layers=[64], tasks={**TASKS_FACE, **TASKS_HAIR})
            }
        )
        
    raise ValueError(f"Blueprint inconnu: {blueprint}")

In [7]:
# --- 6. Moteur d'Entraînement ---
TASK_MAPPING = {
    "barbe": (0, 1, 'bin'), "moustache": (1, 2, 'bin'), "lunettes": (2, 3, 'bin'),
    "taille_cheveux": (3, 6, 'multi'), "couleur_cheveux": (6, 11, 'multi')
}

def calculate_loss(outputs, labels, criteria_bin, criteria_multi):
    """Calcule la perte totale combinée."""
    total_loss = 0
    for name, (start, end, mode) in TASK_MAPPING.items():
        if name not in outputs: continue # Sécurité
        
        pred = outputs[name]
        if mode == 'bin':
            target = labels[:, start]
            total_loss += criteria_bin(pred.squeeze(), target)
        else:
            target = torch.argmax(labels[:, start:end], dim=1)
            total_loss += criteria_multi(pred, target)
    return total_loss

def train_one_epoch(model, loader, optimizer, crit_bin, crit_multi):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = calculate_loss(outputs, labels, crit_bin, crit_multi)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(loader)

def evaluate(model, loader, crit_bin, crit_multi):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            outputs = model(images)
            val_loss += calculate_loss(outputs, labels, crit_bin, crit_multi).item()
    return val_loss / len(loader)

In [8]:
# --- 7. Orchestrateur Hyperopt ---
def objective(params):
    with mlflow.start_run(nested=True):
        # 1. Setup Params
        lr = params['lr']
        batch_size = int(params['batch_size'])
        epochs = 10
        
        # 2. Build Model
        filters = [int(params['base_filters']) * (2**i) for i in range(int(params['n_conv']))]
        tree_config = build_tree_config(params)
        
        model = CNN(filters, tree_config).to(Config.DEVICE)
        
        # 3. Logging Info
        clean_params = {k: str(v) for k, v in params.items() if 'tree' not in k}
        mlflow.log_params(clean_params)
        mlflow.log_dict(tree_config.model_dump(), "tree_structure.json")
        
        print(f"🏗️ Testing: {params['blueprint']} | Layers: {params.get('trunk_layers', 'shared')}")

        # 4. Data & Optimizer
        try:
            train_dl, test_dl = get_data_loaders(batch_size)
        except Exception as e:
            print(f"❌ Data Error: {e}")
            return {'loss': float('inf'), 'status': STATUS_OK}
            
        optimizer = optim.Adam(model.parameters(), lr=lr)
        crit_bin = nn.BCEWithLogitsLoss()
        crit_multi = nn.CrossEntropyLoss()

        # 5. Loop
        best_loss = float('inf')
        for epoch in range(epochs):
            train_loss = train_one_epoch(model, train_dl, optimizer, crit_bin, crit_multi)
            val_loss = evaluate(model, test_dl, crit_bin, crit_multi)
            
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            
            if val_loss < best_loss:
                best_loss = val_loss

        # 6. Save & Signature
        model.eval()
        with torch.no_grad():
            dummy_in = torch.randn(1, 3, 64, 64).to(Config.DEVICE)
            dummy_out = {k: v.cpu().numpy() for k, v in model(dummy_in).items()}
            signature = infer_signature(dummy_in.cpu().numpy(), dummy_out)
            
        mlflow.pytorch.log_model(model, "model", signature=signature)
        
        return {'loss': best_loss, 'status': STATUS_OK}

In [ ]:
mlflow.set_tracking_uri(Config.MLFLOW_URI)
mlflow.set_experiment(Config.EXPERIMENT_NAME)

# Paramètres Communs
cnn_space = {
    'lr': hp.loguniform('lr', np.log(1e-4), np.log(1e-2)),
    'batch_size': hp.choice('batch_size', [16, 32, 64]),
    'dropout': hp.uniform('dropout', 0.1, 0.4),
    'n_conv': hp.choice('n_conv', [2, 3, 4, 5, 6]),
    'base_filters': hp.choice('base_filters', [4, 8, 16, 32, 64]),
}

# Espace de Recherche Conditionnel
search_space = hp.choice('blueprint_selector', [
    
    # A. Hierarchical: On teste différentes formes de couches
    {
        **cnn_space,
        'blueprint': 'hierarchical',
        # Listes explicites pour tester des formes (entonnoir vs droit)
        'trunk_layers': hp.choice('h_trunk', [[256], [512], [512, 256]]),
        'visage_layers': hp.choice('h_vis', [[64], [128, 64]]),
        'hair_layers': hp.choice('h_hair', [[128], [256, 128]]),
        
        # Sous-option conditionnelle
        'use_split_hair': hp.choice('h_split_bool', [False, True]),
        'hair_split_layers': hp.choice('h_split_lay', [[32], [64]])
    },

    # B. Shared: Un gros bloc pour tout le monde
    {
        **cnn_space,
        'blueprint': 'shared_trunk',
        'shared_layers': hp.choice('s_layers', [
            [512, 256], 
            [512, 512, 256],
            [1024, 512, 256]
        ])
    }
])

print("🧠 Démarrage de l'optimisation...")
trials = Trials()
best = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50, # Augmenter selon le temps disponible
    trials=trials
)

print("\n🏆 Meilleure Configuration :")
print(best)

In [10]:
# %%
# --- 8. Entraînement d'une Architecture Connue (Mode Standalone) ---

# 1. Définis ici la configuration exacte que tu veux entraîner
# (Copie ici les valeurs de ton 'best' run ou tes choix manuels)
fixed_params = {
    # Hyperparamètres d'entraînement (non spécifiés sur le schéma, valeurs standard)
    'lr': 1e-3,
    'batch_size': 32,
    'dropout': 0.0,  # Pas de dropout explicite sur le dessin

    # 1. Partie CNN (Gauche du dessin)
    # Dessin : Entrée -> Conv(16) -> Pool -> Conv(32) -> Pool
    'n_conv': 2,           # 2 blocs de convolution (les triangles)
    'base_filters': 16,    # Commence à 16 filtres (puis 32)
    
    # 2. Structure de l'Arbre (Droite du dessin)
    'blueprint': 'hierarchical',
    
    # Le Tronc principal "FC"
    'trunk_layers': [256], # Le grand rectangle vertical noté (..., 256)
    
    # Branche Visage (Haut droite)
    # Note : Ton code regroupe barbe/moustache/lunettes sous "visage".
    # Le dessin montre des couches [64] pour chaque. 
    # Avec ce blueprint, elles partageront une couche de 64.
    'visage_layers': [64], 
    
    # Branche Cheveux (Bas droite)
    # Dessin : Tronc -> FC(128) -> Split -> FC(64) -> Sorties
    'hair_layers': [128],  # La couche notée "256, 128"
    'use_split_hair': True,
    'hair_split_layers': [64] # Les couches notées "128, 64"
}

RUN_NAME = "Classification"

with mlflow.start_run(run_name=RUN_NAME):
    print(f"🚀 Démarrage de l'entraînement unique : {fixed_params['blueprint']}")

    # --- A. Préparation ---
    # Récupération des DataLoaders avec la batch_size fixée
    train_dl, test_dl = get_data_loaders(fixed_params['batch_size'])

    # Construction de la config de l'arbre
    tree_config = build_tree_config(fixed_params)

    # Calcul de la liste des filtres conv (ex: [32, 64, 128, 256])
    filters = [fixed_params['base_filters'] * (2**i) for i in range(fixed_params['n_conv'])]

    # Instanciation du modèle
    model = CNN(filters, tree_config).to(Config.DEVICE)
    
    # Optimiseur et Loss
    optimizer = optim.Adam(model.parameters(), lr=fixed_params['lr'])
    crit_bin = nn.BCEWithLogitsLoss()
    crit_multi = nn.CrossEntropyLoss()

    # --- B. Logging MLflow ---
    mlflow.log_params(fixed_params)
    mlflow.log_dict(tree_config.model_dump(), "tree_structure_fixed.json")

    # --- C. Boucle d'Entraînement ---
    epochs = 20 # Tu peux mettre plus d'epochs pour un training final
    best_val_loss = float('inf')

    for epoch in range(epochs):
        # Utilisation de tes fonctions existantes
        train_loss = train_one_epoch(model, train_dl, optimizer, crit_bin, crit_multi)
        val_loss = evaluate(model, test_dl, crit_bin, crit_multi)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        
        # Sauvegarde du meilleur checkpoint localement si besoin
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # torch.save(model.state_dict(), "best_model_local.pth")

    # --- D. Sauvegarde Finale & Signature ---
    model.eval()
    with torch.no_grad():
        # Création d'un exemple d'input pour la signature du modèle
        dummy_in = torch.randn(1, 3, 64, 64).to(Config.DEVICE)
        # On passe le dummy dans le modèle pour récupérer la structure de sortie
        dummy_out = {k: v.cpu().numpy() for k, v in model(dummy_in).items()}
        
        signature = infer_signature(dummy_in.cpu().numpy(), dummy_out)

    # Log du modèle complet dans MLflow
    mlflow.pytorch.log_model(model, "final_model", signature=signature)
    print(f"✅ Entraînement terminé. Modèle sauvegardé sous le run : {RUN_NAME}")

🚀 Démarrage de l'entraînement unique : hierarchical
🏃 View run Classification at: http://localhost:5000/#/experiments/0/runs/bf542690d0804ff98dc18141655d8fe1
🧪 View experiment at: http://localhost:5000/#/experiments/0


FileNotFoundError: [Errno 2] No such file or directory: '../annotations.csv'